# Estado e Memória

Uma chamada ao modelo é isolada. O que existe é o prompt daquela chamada, e tudo que o modelo parece lembrar está escrito ali ou ficou nos pesos durante o treino. Memória, num sistema de agentes, é o nome do conjunto de decisões sobre o que entra nesse prompt.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import json
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from pydantic import BaseModel

from agentkit import LLM, Agent, tool

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=80)
print(llm.model)

## O modelo não guarda nada

A conversa abaixo tem cinco turnos, e o nome do cliente aparece no primeiro. A pergunta do último turno é feita sozinha, como uma chamada nova.

In [ ]:
SYSTEM = {"role": "system", "content": "You are a concise assistant. Answer in one short sentence."}
TURNS = [
    "Oi, meu nome é Rafael e eu coordeno a operação de logística.",
    "Estou avaliando trocar a transportadora do Nordeste.",
    "Os prazos de entrega estão passando de sete dias.",
    "O contrato atual vence em novembro.",
    "Qual é o meu nome?",
]
print(llm.invoke([SYSTEM, {"role": "user", "content": TURNS[-1]}]))

Nada foi esquecido, porque nada foi guardado. A partir daqui, quatro coisas diferentes recebem o nome de memória, e separá-las é o mapa desta aula. A memória paramétrica é o que o treino deixou nos pesos, que ninguém escreve nem apaga em tempo de uso. O estado da execução é a lista de mensagens que o programa mantém enquanto a conversa acontece. A memória de curto prazo é o que cabe na janela desta execução, com as políticas que decidem o que fica quando o histórico cresce. A memória de longo prazo é o que sobrevive ao fim da conversa, guardado num arquivo fora do modelo.

## Memória paramétrica

Parte do que o modelo responde não vem do prompt. O treino ajustou bilhões de parâmetros sobre um corpus enorme, e o que ficou ali é uma representação difusa desse corpus, consultada de graça em toda chamada.

In [ ]:
print(llm.invoke([{"role": "user", "content": "Qual é a capital da Austrália? Responda apenas a cidade."}]))
print(llm.invoke([{"role": "user", "content": "Quando vence o contrato da transportadora do cliente Rafael?"}]))

A primeira resposta veio dos pesos e está certa. A segunda trata de um fato que nunca esteve no corpus de treino, e aqui o modelo reconhece a lacuna. Reconhecer não é garantido: mais adiante, neste mesmo notebook, uma lacuna parecida vira um nome inventado, e a diferença não está no modelo, e sim em quanto o prompt sugere que a resposta deveria estar ali.

Essa memória tem quatro propriedades que decidem o projeto do resto do sistema. Ela é congelada na data do treino, então não conhece nada posterior. Ela não é editável em tempo de uso, e corrigir um fato exigiria treinar de novo. Ela não é auditável, porque não existe registro de onde cada afirmação veio. E ela não distingue o que sabe do que não sabe, o que faz da lacuna uma invenção com a mesma aparência de resposta. Tudo que vem a seguir existe para cobrir essas quatro lacunas.

## Estado da execução

O estado da conversa é uma lista de mensagens mantida pelo programa, e a função abaixo é toda a escrita que ela exige.

In [ ]:
def add_message(history: list, role: str, content: str) -> None:
    """Acrescenta uma mensagem ao histórico, no lugar."""
    history.append({"role": role, "content": content})

In [ ]:
history = [SYSTEM]
for turn in TURNS:
    add_message(history, "user", turn)
    add_message(history, "assistant", llm.invoke(history))
print(history[-1]["content"])

Com o histórico inteiro no prompt, a resposta está correta. O preço é que cada turno reenvia tudo que veio antes, e o prompt cresce a cada rodada.

In [ ]:
growth = []
for size in range(2, len(history) + 1, 2):
    prompt = llm.tokenizer.apply_chat_template(
        history[:size], tokenize=False, add_generation_prompt=True
    )
    growth.append({"messages": size, "tokens": len(llm.tokenizer.encode(prompt))})
pd.DataFrame(growth)

A conta é a mesma para qualquer conversa: o custo de um turno é proporcional ao tamanho do histórico, e o total de uma conversa de $n$ turnos cresce com o quadrado de $n$.

## Memória de curto prazo

O histórico completo resolve a conversa e não escala: em algum ponto ele não cabe na janela de contexto, e antes disso já ficou caro. As duas políticas abaixo decidem o que fica.

### Janela deslizante

A política mais simples corta pelo fim, mantendo as últimas mensagens.

In [ ]:
def recent_messages(history: list, max_messages: int = 6) -> list:
    """Devolve as últimas mensagens do histórico."""
    # O corte é por número de mensagens, nunca por turnos: o histórico passa a ter
    # observações de ferramenta, e qualquer cálculo que assuma alternância quebra.
    return history[-max_messages:] if max_messages > 0 else []

A conversa é refeita com uma janela de quatro mensagens. Antes de rodar, tente prever a resposta da última pergunta.

In [ ]:
windowed = [SYSTEM]
for turn in TURNS:
    add_message(windowed, "user", turn)
    add_message(windowed, "assistant", llm.invoke([SYSTEM] + recent_messages(windowed, 4)))
print(windowed[-1]["content"])

A informação decisiva estava no primeiro turno e saiu da janela quatro mensagens depois. O corte é cego: ele olha para a posição e não para o conteúdo, e descarta com a mesma facilidade uma saudação e o nome do cliente.

### Sumarização progressiva

A alternativa é comprimir o trecho antigo em vez de descartá-lo, usando o próprio modelo.

In [ ]:
def summarize_messages(messages: list) -> str:
    """Resume um trecho de conversa em poucas linhas."""
    # As mensagens viram texto no estilo papel: conteúdo. Passar str(messages)
    # injetaria a representação Python, com chaves e aspas, dentro do prompt.
    conversation = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    return llm.invoke([
        {"role": "system", "content": (
            "Você resume conversas de atendimento para outro atendente continuar."
            " Escreva em português, em frases completas e afirmativas, preservando"
            " nomes, números e prazos. Comece por: O usuário é ..."
        )},
        {"role": "user", "content": f"Resuma a conversa abaixo em no máximo três linhas.\n\n{conversation}"},
    ], max_tokens=120)


summary = summarize_messages(history[1:7])
print(summary)

A instrução do resumo decide o que sobrevive, e não só quais fatos: a forma também conta. Pedir frases completas começando por quem é o usuário produz um texto que o modelo consegue ligar à conversa; sem isso ele devolve fragmentos do tipo nome seguido de função, que valem como anotação e se soltam do resto.

A perda é irreversível. A janela deslizante descarta mensagens que ainda existem na lista, e nada impede o programa de voltar a buscá-las; o resumo substitui o trecho por uma paráfrase, e o que ficou de fora não volta mais, em nenhuma pergunta futura. O uso prático combina as duas políticas: o trecho antigo vira resumo e os turnos recentes entram inteiros.

In [ ]:
def compress(history: list, keep: int = 4) -> list:
    """Substitui o trecho antigo do histórico por um resumo e mantém o fim intacto."""
    if len(history) <= keep + 1:
        return history
    older, recent = history[1:-keep], history[-keep:]
    summary = summarize_messages(older)
    # O resumo entra na mensagem de sistema que já existe. Como segunda mensagem
    # de sistema ele é lido como texto solto, e o modelo deixa de ligá-lo à conversa.
    return [
        {"role": "system", "content": f"{history[0]['content']}\n\nResumo da conversa anterior:\n{summary}"},
        *recent,
    ]


compressed = compress(history)
print(llm.invoke(compressed + [{"role": "user", "content": "Qual é o meu nome?"}], max_tokens=60))

A resposta continua certa com um prompt bem menor, e o detalhe que faz isso funcionar está no comentário do `compress`: o resumo entra dentro da mensagem de sistema que já existe. Colocado como uma segunda mensagem de sistema, ou como turno de usuário, o mesmo texto passa a ser lido como material solto, e a pergunta sobre o próprio nome volta a ser respondida com um nome inventado. Onde o contexto entra pesa tanto quanto o que ele diz.

In [ ]:
print(llm.invoke(
    compressed + [{"role": "user", "content": "Me lembre: quem sou eu e qual empresa quero trocar?"}],
    max_tokens=80,
))

full_tokens = len(llm.tokenizer.encode(llm.tokenizer.apply_chat_template(history, tokenize=False)))
short_tokens = len(llm.tokenizer.encode(llm.tokenizer.apply_chat_template(compressed, tokenize=False)))
print(f"histórico completo: {full_tokens} tokens")
print(f"histórico comprimido: {short_tokens} tokens")

Com a pergunta mais próxima da forma do resumo, a mesma informação volta inteira. Comprimir muda o formato do que ficou guardado, e o formato decide quais perguntas ainda têm resposta. O ganho é o prompt menor em todas as chamadas seguintes, ao custo de uma chamada extra ao modelo, o que só se paga em conversas longas.

### Exercício 1

Resumir custa uma chamada a mais e economiza tokens em todas as chamadas seguintes. Meça a partir de quando isso compensa.

Rode a conversa longa abaixo e, para cada turno, calcule duas contas: os tokens do prompt com o histórico inteiro e os tokens do prompt comprimido, somando também o que a chamada de resumo gastou, que está em `llm.last_usage`. Decida de quantos em quantos turnos o resumo é refeito, porque essa escolha muda o resultado.

Monte um `DataFrame` com o turno e o custo acumulado das duas políticas, e responda em que turno a compressão passa a sair mais barata, e o que acontece com esse ponto quando a janela mantida dobra de tamanho.

In [ ]:
LONG_TURNS = [
    "Oi, meu nome é Rafael e eu coordeno a operação de logística.",
    "Estou avaliando trocar a transportadora do Nordeste.",
    "Os prazos de entrega estão passando de sete dias.",
    "O contrato atual vence em novembro.",
    "O orçamento aprovado para frete é de 120 mil reais por mês.",
    "A maior parte dos pedidos sai de Recife.",
    "Tivemos três reclamações de atraso na semana passada.",
    "O time comercial pediu um relatório semanal.",
    "A diretoria quer reduzir o custo por entrega em 8%.",
    "Apareceu uma proposta de transportadora nova, sem cobertura no interior.",
    "Preciso decidir até o fim do mês.",
    "Qual é o meu nome e qual é o meu orçamento de frete?",
]

# Seu código aqui

## Memória de longo prazo

Tudo acima morre com a lista de mensagens, e a lista morre com o processo. Guardar algo para a próxima execução exige uma estrutura fora do modelo e fora da memória do programa, e o mínimo que serve é um arquivo. O JSON abaixo é a memória inteira: o aluno pode abri-lo em qualquer editor e ver o estado do agente.

In [ ]:
MEMORY_PATH = Path("memory.json")


def load_memory() -> dict:
    """Lê a memória do disco, ou devolve uma vazia na primeira execução."""
    if MEMORY_PATH.exists():
        return json.loads(MEMORY_PATH.read_text(encoding="utf-8"))
    return {"facts": {}, "episodes": []}


def save_memory(memory: dict) -> None:
    """Grava a memória inteira no disco, em JSON legível."""
    MEMORY_PATH.write_text(json.dumps(memory, ensure_ascii=False, indent=2), encoding="utf-8")


# O arquivo é apagado aqui para a saída da aula não depender de execuções
# anteriores. Num sistema real ele é exatamente o que não se apaga.
MEMORY_PATH.unlink(missing_ok=True)
memory = load_memory()
memory

### Fatos semânticos

A memória semântica guarda o que é verdade sobre o cliente, o domínio e as regras do negócio, sem registro de quando foi aprendido. A estrutura é um dicionário de campo para valor, e essa escolha decide sozinha um problema que uma lista de anotações deixaria em aberto: gravar num campo que já existe substitui o valor, então o fato do cliente tem uma versão só, sempre a última.

In [ ]:
def set_fact(memory: dict, field: str, value: str) -> None:
    """Grava um fato. Gravar num campo que já existe substitui o valor."""
    memory["facts"][field] = value


set_fact(memory, "nome", "Rafael")
set_fact(memory, "cargo", "coordenador de logística")
set_fact(memory, "orcamento_frete", "120 mil reais por mês")
set_fact(memory, "vencimento_contrato", "novembro de 2026")
set_fact(memory, "status_proposta", "aguardando resposta do cliente")
save_memory(memory)
print(MEMORY_PATH.read_text(encoding="utf-8"))

Escrever os fatos à mão não escala. O que vale guardar aparece na conversa, e extraí-lo é uma tarefa de saída estruturada, como na aula anterior: um esquema entra, um objeto validado sai.

In [ ]:
class Fact(BaseModel):
    field: str
    value: str


class Facts(BaseModel):
    facts: list[Fact]


conversation_text = "\n".join(f"{m['role']}: {m['content']}" for m in history[1:9])
extracted = llm.generate_structured(
    [{"role": "user", "content": f"Extract durable facts about the user.\n\n{conversation_text}"}],
    Facts,
    max_tokens=200,
)
pd.DataFrame([fact.model_dump() for fact in extracted.facts])

O esquema livre autoriza qualquer campo, e o modelo usa essa liberdade. Os nomes saem em inglês, uma região vira nome de transportadora, e um campo que a conversa não afirma é preenchido com um valor de recheio em vez de ficar vazio. O pior nem é o conteúdo: é que as chaves são inventadas a cada chamada, então a extração da semana que vem não bate com a desta, e atualizar deixa de ser possível.

A política de escrita é uma decisão de projeto, e o esquema é onde ela se declara. Fixar os campos resolve as três coisas de uma vez: o vocabulário fica estável, o conjunto fica limitado ao que este sistema acompanha, e `str | None` autoriza a ausência, com a regra de preenchimento na mensagem de sistema.

In [ ]:
class ClientFacts(BaseModel):
    nome: str | None
    cargo: str | None
    regiao: str | None
    prazo_entrega: str | None
    vencimento_contrato: str | None
    telefone: str | None


profile = llm.generate_structured([
    {"role": "system", "content": (
        "Extraia fatos duráveis sobre o usuário. Use null em todo campo que a"
        " conversa não afirmar explicitamente. Não deduza e não invente."
    )},
    {"role": "user", "content": conversation_text},
], ClientFacts, max_tokens=200)
print(profile)

for field, value in profile.model_dump().items():
    if value:
        set_fact(memory, field, value)
save_memory(memory)
memory["facts"]

O campo que já existia foi reescrito com o que a conversa afirma, os campos novos entraram, e `telefone` ficou de fora porque voltou nulo, que é o que a regra de preenchimento pediu. Nada disso exigiu uma decisão explícita de atualização: quem fez o trabalho foi a chave do dicionário.

Nessa escala não existe recuperação, e vale dizer isso com todas as letras. Com algumas dezenas de fatos, a tabela inteira entra no prompt e custa algumas centenas de tokens, o que é mais barato e mais confiável do que qualquer busca. Escolher o que trazer só passa a ser um problema quando o volume não cabe na janela, e é aí que a Unidade II começa.

In [ ]:
def facts_block(memory: dict) -> str:
    """Escreve todos os fatos como texto, para entrar no prompt de uma vez."""
    return "\n".join(f"- {field}: {value}" for field, value in memory["facts"].items())


print(llm.invoke([
    {"role": "system", "content": f"{SYSTEM['content']}\n\nFatos conhecidos sobre o usuário:\n{facts_block(memory)}"},
    {"role": "user", "content": "Quanto posso gastar por mês com frete?"},
], max_tokens=60))

### Memória episódica

A memória episódica guarda o que aconteceu, com quando aconteceu. Ela responde a outra pergunta, que é o que já foi feito, e por isso os itens carregam data e não se substituem: um fato semântico continua valendo até ser trocado, enquanto um episódio vale para sempre como registro e pode estar obsoleto como informação. A estrutura muda junto com a pergunta: aqui é uma lista, e ela cresce sem limite.

In [ ]:
def add_episode(memory: dict, when: str, text: str) -> None:
    """Acrescenta um registro datado à memória episódica."""
    # hits e accessed_at são gravados no acesso e usados depois pela poda.
    memory["episodes"].append(
        {"when": when, "text": text, "hits": 0, "accessed_at": time.time()}
    )


EPISODES = [
    ("2026-03-02", "Reunião de renovação: o cliente pediu desconto de 10% e ficou de responder até sexta."),
    ("2026-04-18", "Enviamos a proposta revisada por e-mail, sem resposta até agora."),
    ("2026-05-27", "O cliente reclamou de atraso na entrega de Recife e pediu relatório semanal."),
    ("2026-06-10", "Revisamos a tabela de preços com o time comercial."),
]
for when, text in EPISODES:
    add_episode(memory, when, text)
save_memory(memory)
pd.DataFrame(memory["episodes"])[["when", "text"]]

Como a lista cresce, ela precisa de um filtro. Os dois filtros óbvios são a data, que a estrutura já tem, e as palavras em comum entre a consulta e o registro.

In [ ]:
def words(text: str) -> set[str]:
    """Palavras com mais de três letras, em minúsculas e sem acento."""
    plain = unicodedata.normalize("NFKD", text.lower()).encode("ascii", "ignore").decode()
    return {word for word in re.findall(r"[a-z0-9]+", plain) if len(word) > 3}


def find_episodes(memory: dict, query: str = "", since: str = "", k: int = 2) -> list[dict]:
    """Filtra episódios por data e por palavras em comum com a consulta."""
    query_words = words(query)
    found = []
    for episode in memory["episodes"]:
        if episode["when"] < since:
            continue
        shared = query_words & words(episode["text"])
        if query and not shared:
            continue
        found.append((len(shared), episode))
    found.sort(key=lambda item: (item[0], item[1]["when"]), reverse=True)
    results = []
    for _, episode in found[:k]:
        episode["hits"] += 1
        episode["accessed_at"] = time.time()
        results.append(episode)
    return results


for episode in find_episodes(memory, since="2026-05-01"):
    print("data:   ", episode["when"], "|", episode["text"])
for episode in find_episodes(memory, query="O cliente reclamou de alguma entrega?"):
    print("palavras:", episode["when"], "|", episode["text"])

O filtro por data é exato e o filtro por palavras é uma aproximação grosseira: o registro certo vem na frente porque repete três palavras da pergunta, e junto vem um registro de março que só compartilha a palavra cliente. A última seção do notebook mostra o caso em que essa aproximação falha por completo.

### Memória procedural

A memória procedural guarda como fazer. São os procedimentos, as convenções e as preferências de trabalho, e eles entram no prompt como instrução de sistema, e não como dado a consultar.

In [ ]:
PROCEDURES = [
    "Para renovar um contrato: conferir o prazo, pedir aprovação do gestor e enviar a minuta em PDF.",
    "Reclamações de atraso são respondidas no mesmo dia, com o número do pedido no assunto.",
    "Relatórios vão por e-mail toda segunda de manhã, nunca por telefone.",
]

print(llm.invoke([
    {"role": "system", "content": (
        f"{SYSTEM['content']}\n\nProcedimentos a seguir:\n"
        + "\n".join(f"- {procedure}" for procedure in PROCEDURES)
    )},
    {"role": "user", "content": "Vou renovar o contrato do Rafael. Por onde começo?"},
], max_tokens=80))

O procedimento entrou como instrução e a resposta passou a segui-lo. Essa é a diferença de tratamento entre os três tipos: o fato semântico e o episódio entram como dado, para o modelo consultar, e o procedimento entra como instrução, para o modelo obedecer. Repare também que os procedimentos não estão no `memory.json`: eles não são aprendidos com o usuário, são escritos pela equipe e versionados junto com o código.

## Escrita e esquecimento

Ler a memória é a parte fácil. O problema é escrever, porque toda informação nova chega em conflito com o que já está guardado, e existem quatro decisões possíveis: acrescentar um campo que não existia, atualizar o valor de um campo que existia, apagar um campo que deixou de valer, e ignorar o que já está registrado.

In [ ]:
def apply_fact(memory: dict, field: str, value: str) -> str:
    """Aplica um fato novo e devolve qual das quatro decisões foi tomada."""
    current = memory["facts"].get(field)
    if not value:
        memory["facts"].pop(field, None)
        return "apagar"
    if current is None:
        memory["facts"][field] = value
        return "adicionar"
    if current == value:
        return "ignorar"
    memory["facts"][field] = value
    return "atualizar"


CANDIDATES = [
    ("vencimento_contrato", "dezembro de 2026"),
    ("nome", "Rafael"),
    ("cidade_origem", "Recife"),
    ("status_proposta", ""),
]
decisions = pd.DataFrame([
    {"campo": field, "antes": memory["facts"].get(field), "novo": value,
     "decisão": apply_fact(memory, field, value)}
    for field, value in CANDIDATES
])
save_memory(memory)
decisions

As quatro decisões existem em qualquer sistema de memória, e o que muda entre projetos é quem as toma. Numa lista de anotações em texto, atualizar é um problema em aberto: o valor novo entra como mais uma linha, a linha antiga continua lá, e alguma etapa de leitura precisa decidir qual das duas vale. Aqui a atualização é propriedade da estrutura, porque o campo comporta um valor só.

O que a estrutura não decide é o mapeamento: qual campo uma frase do usuário afeta, e se ela afeta algum. Essa parte continua sendo do modelo, e é por isso que a ferramenta de escrita da próxima seção recebe o campo como argumento.

### Esquecimento

Os fatos não crescem sem limite, porque o conjunto de campos é limitado pelo esquema. Os episódios crescem: um por atendimento, para sempre. A poda combina recência e uso, com uma pontuação que decai com o tempo desde o último acesso e cresce com o número de acessos, de modo que um registro antigo e muito consultado sobrevive.

In [ ]:
def forget_episodes(memory: dict, max_items: int = 3, half_life: float = 3600.0) -> int:
    """Poda a memória episódica por recência de acesso e uso, e devolve quantos saíram."""
    if len(memory["episodes"]) <= max_items:
        return 0
    now = time.time()
    ranked = sorted(
        memory["episodes"],
        key=lambda episode: (1 + episode["hits"]) * 0.5 ** ((now - episode["accessed_at"]) / half_life),
        reverse=True,
    )
    removed = len(memory["episodes"]) - max_items
    memory["episodes"] = sorted(ranked[:max_items], key=lambda episode: episode["when"])
    return removed


# Num notebook que roda em minutos nada envelhece: o carimbo de um episódio é
# empurrado dez horas para trás para a decadência ficar visível.
memory["episodes"][1]["accessed_at"] = time.time() - 10 * 3600

print(f"episódios antes: {len(memory['episodes'])}, removidos: {forget_episodes(memory)}")
save_memory(memory)
pd.DataFrame(memory["episodes"])[["when", "hits", "text"]]

Saiu o registro que ninguém consultou e que estava com o carimbo mais antigo. A poda nunca olhou o conteúdo, e isso é uma política discutível de propósito: um registro raro e crítico, consultado uma vez por ano, é exatamente o que ela descarta. Alternativas comuns são orçamentos separados por tipo, já que episódios envelhecem e fatos não, e marcar itens como permanentes na escrita.

### Exercício 2

Escreva um episódio antigo que contradiz um fato atual, por exemplo um desconto de 10% concedido em março e um campo `politica_desconto` dizendo que hoje o máximo é 5% sem aprovação do diretor.

Monte três prompts para a pergunta "que desconto podemos dar para este cliente?": um só com os fatos, um só com os episódios e um com os dois. Mostre as três respostas e responda qual delas você entregaria ao usuário, e o que precisaria estar guardado para que essa escolha não dependesse de você.

In [ ]:
# Seu código aqui

## Memória no agente

A sessão nova começa com o histórico vazio, e o que atravessa a fronteira é o arquivo. Num agente isso tem dois lados: o que ele lê da memória entra no prompt antes da primeira chamada, e o que ele aprende na conversa volta para o arquivo por uma ferramenta.

In [ ]:
def memory_messages(question: str) -> list[dict]:
    """Monta a mensagem de sistema com o que a memória já sabe."""
    content = f"{SYSTEM['content']}\n\nFatos conhecidos sobre o usuário:\n{facts_block(memory)}"
    episodes = find_episodes(memory, question, k=1)
    if episodes:
        content += f"\n\nRegistro recente:\n- {episodes[0]['when']}: {episodes[0]['text']}"
    return [{"role": "system", "content": content}]


@tool
def save_fact(field: str, value: str) -> str:
    """Grava um fato sobre o usuário na memória de longo prazo, um campo por chamada."""
    set_fact(memory, field, value)
    save_memory(memory)
    return f"gravado: {field} = {value}"


agent = Agent(llm, [save_fact], max_steps=3)

In [ ]:
question = "Quanto posso gastar por mês com frete?"

print(llm.invoke([SYSTEM, {"role": "user", "content": question}], max_tokens=60))

In [ ]:
for message in agent.run(memory_messages(question) + [{"role": "user", "content": question}]):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

A resposta mudou porque o prompt mudou. O modelo continua sem memória nenhuma além da paramétrica: o que existe é um arquivo do lado de fora e uma decisão de qual trecho dele entra no contexto.

O outro lado é a escrita. O pedido abaixo traz um valor novo para um campo que já está guardado, e a única forma de ele sobreviver ao fim da conversa é o agente chamar a ferramenta.

In [ ]:
request = "Anote que o orçamento de frete subiu para 150 mil reais por mês."

for message in agent.run(memory_messages(request) + [{"role": "user", "content": request}]):
    print(message["role"], ":", message.get("content") or message["tool_calls"])

In [ ]:
del memory
memory = load_memory()
memory["facts"]

O dicionário do notebook foi apagado e a memória voltou do disco com o valor novo, e com um valor só para o campo. Numa memória feita de anotações em texto, o mesmo pedido teria deixado duas frases sobre orçamento convivendo, e a próxima leitura precisaria escolher entre elas sem nenhum critério além da ordem. Aqui a escrita e a atualização são a mesma operação.

## O limite da busca literal

Falta um problema, e ele é o assunto da próxima unidade. O registro abaixo entra na memória episódica com o vocabulário de quem escreveu, e a pergunta chega com o vocabulário de quem consulta.

In [ ]:
add_episode(memory, "2026-07-04", "Aprovamos o estorno da nota 8842; o crédito entra na fatura seguinte, em até 72 horas úteis.")
save_memory(memory)

question = "Em quantos dias o dinheiro volta para o cliente?"
print("palavras em comum:", words(question) & words(memory["episodes"][-1]["text"]))
for episode in find_episodes(memory, question):
    print(episode["when"], "|", episode["text"])

A interseção de palavras é vazia: as duas frases falam da mesma coisa e não têm uma palavra em comum. O filtro não devolve nada de útil e ainda devolve o registro errado, porque a palavra cliente aparece num episódio que não tem relação com estorno. Nenhum ajuste de limiar conserta isso, e uma lista de sinônimos escrita à mão conserta um caso por vez.

O que falta é comparar significado em vez de comparar strings, e a forma de fazer isso é representar cada texto como um vetor em que frases próximas em sentido ficam próximas no espaço. É esse mecanismo, e o sistema de recuperação construído sobre ele, que abre a Unidade II. Enquanto os fatos couberem inteiros no prompt, como couberam neste notebook, o problema não aparece; ele aparece na escala em que a seleção passa a ser obrigatória.

### Exercício 3

Reproduza a falha com um caso seu. Guarde um episódio ou uma regra escrita com um vocabulário e faça a pergunta com outro, sem repetir nenhuma palavra do registro, por exemplo guardar um prazo de troca de produto e perguntar até quando dá para devolver a compra.

Mostre a interseção de palavras e o resultado de `find_episodes`. Depois tente consertar por dentro da busca literal, acrescentando sinônimos ao texto guardado ou à consulta, e responda duas coisas: quantos sinônimos foram necessários para o seu caso, e por que essa correção não sobrevive ao próximo usuário que perguntar de um jeito que você não previu.

In [ ]:
# Seu código aqui